# Validasi Prioritas

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
BASE = ROOT
OUT = ROOT / "hasil_validasi_prioritas"
OUT.mkdir(parents=True, exist_ok=True)
if not (ROOT / "hasil_integrasi_kesehatan" / "01_aqms_harian_kota.csv").is_file():
    raise FileNotFoundError("Jalankan integrasi_polusi_kesehatan.ipynb terlebih dahulu.")


In [ ]:
a = pd.read_csv(BASE / "hasil_integrasi_kesehatan/01_aqms_harian_kota.csv")
h = pd.read_csv(BASE / "hasil_integrasi_kesehatan/02_kesehatan_harian_kota.csv")
print("AQMS:",a.shape,"Kesehatan:",h.shape)


In [ ]:
for d in (a,h):
    d["prov_key"] = d["prov_key"].str.strip().str.lower()
    d["kota_key"] = d["kota_key"].astype("string").str.strip().str.lower().str.replace(r"^kota administrasi ", "", regex=True)
    d["tanggal"] = pd.to_datetime(d["tanggal"])
    assert not d.duplicated(["prov_key","kota_key","tanggal"]).any()


## Gabungkan data

In [ ]:
joined = h.merge(a,on=["prov_key","kota_key","tanggal"],how="left",indicator=True,validate="one_to_one")
joined["beban_balita_lansia"] = joined["resp_admissions_under5"] + joined["resp_admissions_over60"]
print(joined["_merge"].value_counts())


## Cakupan PM2.5

In [ ]:
rows=[]
for (p,k),g in joined.groupby(["prov_key","kota_key"]):
    valid=g[g["pm25_ug_m3"].notna() & g["beban_balita_lansia"].notna()]
    rows.append(dict(provinsi=p,kota_kabupaten=k,hari_kesehatan=len(g),hari_aqms_tercocok=int(g["_merge"].eq("both").sum()),hari_pm25_valid=len(valid),cakupan_pm25_pct=round(100*len(valid)/len(g),2),pm25_rerata_hari_sepadan=valid.pm25_ug_m3.mean(),kasus_balita_lansia_rerata_hari_sepadan=valid.beban_balita_lansia.mean(),total_kasus_balita_lansia_hari_sepadan=valid.beban_balita_lansia.sum(),status="Layak skor eksploratif" if len(valid)>=30 else "Tidak cukup data PM2.5"))
res=pd.DataFrame(rows)
print(res.status.value_counts())


## Skor prioritas

In [ ]:
eligible=res.status.eq("Layak skor eksploratif")
if not eligible.any():
    print("Belum ada wilayah yang memenuhi syarat minimal 30 hari PM2.5.")
res["persentil_pm25"]=np.nan;res["persentil_kasus"]=np.nan
res.loc[eligible,"persentil_pm25"]=res.loc[eligible,"pm25_rerata_hari_sepadan"].rank(pct=True)
res.loc[eligible,"persentil_kasus"]=res.loc[eligible,"kasus_balita_lansia_rerata_hari_sepadan"].rank(pct=True)
res["skor_prioritas_eksploratif"]=((res.persentil_pm25+res.persentil_kasus)/2*100).round(2)
res["peringkat_eksploratif"]=res.skor_prioritas_eksploratif.rank(ascending=False,method="min").astype("Int64")
med_pm=res.loc[eligible,"pm25_rerata_hari_sepadan"].median();med_case=res.loc[eligible,"kasus_balita_lansia_rerata_hari_sepadan"].median()
def cat(r):
    if pd.isna(r.skor_prioritas_eksploratif):return "Belum dapat dinilai"
    p=r.pm25_rerata_hari_sepadan>=med_pm;c=r.kasus_balita_lansia_rerata_hari_sepadan>=med_case
    return "Keduanya relatif tinggi" if p and c else "PM2.5 relatif tinggi" if p else "Kasus relatif tinggi" if c else "Keduanya relatif rendah"
res["kategori_prioritas"]=res.apply(cat,axis=1)
res=res.sort_values(["peringkat_eksploratif","provinsi","kota_kabupaten"],na_position="last")
display(res.head(10))


In [ ]:
res.to_csv(OUT/"01_prioritas_tervalidasi.csv",index=False)
joined["_merge"]=joined["_merge"].astype(str)
audit=joined.groupby(["prov_key","kota_key"],as_index=False).agg(hari_total=("tanggal","size"),hari_aqms_cocok=("_merge",lambda x:(x=="both").sum()),hari_pm25_terisi=("pm25_ug_m3","count"))
audit.to_csv(OUT/"02_audit_integrasi_per_wilayah.csv",index=False)
joined.groupby("tanggal",as_index=False).agg(jumlah_wilayah_kesehatan=("kota_key","size"),jumlah_wilayah_aqms_cocok=("_merge",lambda x:(x=="both").sum())).to_csv(OUT/"03_audit_cakupan_harian.csv",index=False)
res[["provinsi","kota_kabupaten","pm25_rerata_hari_sepadan","kasus_balita_lansia_rerata_hari_sepadan","cakupan_pm25_pct","skor_prioritas_eksploratif","kategori_prioritas","peringkat_eksploratif","status"]].to_csv(OUT/"04_dashboard_priority_final.csv",index=False)
print("File disimpan di",OUT)
